## Music Recommendation
### University of Virginia
### DS 5110: Big Data Systems
### Last Updated: March 18, 2026


---

#### Instructions

In this assignment, you will build an ALS recommendation algorithm based on user listening data from Autoscrobbler.

The data consists of: 
- user data (listeners)
- item data (songs)
- interaction data (user listens, which is implicit feedback).  

Additional details are in the `README` file. Be sure to review in advance.

Develop/test code to accomplish the tasks below, clearly showing the results and documenting your work.

You will use the **DataFrame API** for all work.

**TOTAL POINTS: 10**
***

#### Preprocessing Steps in order:

1) To reduce runtime, you will sample 20% of the interaction data from `user_artist_data.txt` using seed=314 (and use this seed throughout the lab)
2) Next, remove users that listened to fewer than 10 **artists** (you may ignore aliasing in this step)
3) Lastly, split the data: 60% for training / 20% for validation / 20% for testing

In [1]:
# ----------------------------------------------------------------------------------------------------------------------------------------------
# PREPROCESSING STEP 1
# To reduce runtime, you will sample 20% of the interaction data from user_artist_data.txt using seed=314 (and use this seed throughout the lab)
# ----------------------------------------------------------------------------------------------------------------------------------------------

# Before doing anything, create a Spark session.
from pyspark.sql import SparkSession
spark = SparkSession.builder.getOrCreate()

# Define the schema for the file to import.  
# -- I added this after reviewing the data file and running an initial import.
# -- Even though I saw the file had no headers, I didn't initially stop to think I needed to add them!  :)
from pyspark.sql.types import StructType, StructField, IntegerType
autoscrobbler_schema = StructType([
                                    StructField("userid", IntegerType()),
                                    StructField("artistid", IntegerType()),
                                    StructField("playcount", IntegerType()),
                                    ])
     

# Load the data.
# The columns are separated by a space, so need to include sep=" ".
df_user = spark.read.csv("/standard/ds7200-apt4c/large_datasets/user_artist_data.txt",  sep=" ", header=False, schema=autoscrobbler_schema)

# Check to see if any columns have null values.
df_user.createOrReplaceTempView("df_user")
df_user_null = spark.sql("""
    SELECT COUNT(*) AS RowsWithNulls
    FROM df_user
    WHERE 
        userid IS NULL 
        OR artistid IS NULL
        OR playcount IS NULL
""")
print(f"""There are {df_user_null.show()} rows with NULL values.""")

# Since no rows have null values, continue...
# Select a 20% sample from the data.
df_user_small = df_user.sample(fraction=0.20, seed=314)

# Confirm rows were imported.
print(f"""There are {df_user_small.count()} rows in the imported data set.""")

# Review a few records and the schema to make sure it was applied correctly.
# And it does, so yay!
df_user.show(5)
df_user.printSchema()

Using Spark's default log4j profile: org/apache/spark/log4j2-defaults.properties
Setting default log level to "WARN".
To adjust logging level use sc.setLogLevel(newLevel). For SparkR, use setLogLevel(newLevel).
26/10/11 19:28:32 WARN NativeCodeLoader: Unable to load native-hadoop library for your platform... using builtin-java classes where applicable
                                                                                

+-------------+
|RowsWithNulls|
+-------------+
|            0|
+-------------+

There are None rows with NULL values.


There are 4859650 rows in the imported data set.
+-------+--------+---------+
| userid|artistid|playcount|
+-------+--------+---------+
|1000002|       1|       55|
|1000002| 1000006|       33|
|1000002| 1000007|        8|
|1000002| 1000009|      144|
|1000002| 1000010|      314|
+-------+--------+---------+
only showing top 5 rows
root
 |-- userid: integer (nullable = true)
 |-- artistid: integer (nullable = true)
 |-- playcount: integer (nullable = true)



In [2]:
# ----------------------------------------------------------------------------------------------------------------------------------------------
# PREPROCESSING STEP 2
# Next, remove users that listened to fewer than 10 artists (you may ignore aliasing in this step)
# ----------------------------------------------------------------------------------------------------------------------------------------------

# Remove users that listen to fewer than 10 artists using SQL.
# -- First register the DF as a temp view first.
df_user_small.createOrReplaceTempView("df_user_small")

# See how may users have listened to fewer than 10 artists.

# Select the columns needed for this analysis.
df_user_go = spark.sql("""
    SELECT *
    FROM df_user_small
    WHERE userid IN
        (
            SELECT userid
            FROM df_user_small
            GROUP BY userid
            HAVING COUNT(DISTINCT artistid) > 9
        )
""")

print(f"""There are {df_user_go.count()} rows after removing users that listened to less than 10 artists.""")

[Stage 14:================================>                        (8 + 6) / 14]

There are 4684468 rows after removing users that listened to less than 10 artists.


In [3]:
# ----------------------------------------------------------------------------------------------------------------------------------------------
# PREPROCESSING STEP 3
# Lastly, split the data: 60% for training / 20% for validation / 20% for testing
# ----------------------------------------------------------------------------------------------------------------------------------------------
# Split the dataset.
trainDF, valDF, testDF = df_user_go.randomSplit([.6, .2, .2], seed=314)

# Cache the datasets for better downstream processing.
trainDF = trainDF.cache()
valDF = valDF.cache()
testDF = testDF.cache()

print(f"""There are {trainDF.count()} rows in the training set, {valDF.count()} rows in the validation set, and {testDF.count()} in the test set.""")

[Stage 61:===========================================>         (165 + 18) / 200]

There are 2810567 rows in the training set, 936569 rows in the validation set, and 937332 in the test set.


In [4]:
# TASK 1 (1 PT)
# Show the first row from the training set
trainDF.show(1)

# Result should remain consistent with seed=314.
#  |userid|artistid|playcount|
#  +------+--------+---------+
#  |   321|      40|        1|
#  +------+--------+---------+

+------+--------+---------+
|userid|artistid|playcount|
+------+--------+---------+
|  7340|     930|        1|
+------+--------+---------+
only showing top 1 row


In [5]:
# TASK 2 (1 PT)
# Show the first row from the test set. We will refer to this user as Test User 1.
testDF.show(1)

# Result should remain consistent with seed=314.
#  +------+--------+---------+
#  |userid|artistid|playcount|
#  +------+--------+---------+
#  |   321|     357|        3|
#  +------+--------+---------+

+------+--------+---------+
|userid|artistid|playcount|
+------+--------+---------+
|  7340|    1835|        2|
+------+--------+---------+
only showing top 1 row


In [6]:
# ALS MODEL 1: Train the ALS implicit model on the training data (since the measurements are activity and not ratings)
# Use parameters: rank=8, maxIter=5, regParam=0.02, alpha=0.01

# The parameters above do not align to the old MLlib library, so using the newer ML library.
from pyspark.ml.recommendation import ALS

# Create the model specification using the parameters for ML.
als_spec = ALS(
    userCol="userid",
    itemCol="artistid",
    ratingCol="playcount",
    implicitPrefs=True,
    coldStartStrategy="drop",
    rank=8,
    maxIter=5,
    regParam=0.02,
    alpha=0.01
)

# Fit the model.
als_model = als_spec.fit(trainDF)

print("The model has been trained.")

26/10/11 19:29:08 WARN InstanceBuilder: Failed to load implementation from:dev.ludovic.netlib.blas.JNIBLAS
26/10/11 19:29:08 WARN InstanceBuilder: Failed to load implementation from:dev.ludovic.netlib.lapack.JNILAPACK


The model has been trained.


In [7]:
# TASK 3 (2 PTS)
# Using ALS MODEL 1, make 10 recommendations for Test User 1
als_predictions = als_model.transform(testDF)

# Use SQL to get 10 predictions for Test User 1 (userid = 321).
als_predictions.createOrReplaceTempView("als_predictions")

# STEP 1: First, get the predictions for userid = 321.
test_user1 = spark.sql("""
    SELECT *
    FROM als_predictions
    WHERE 
        userid=321
        AND NOT ISNAN(prediction)
""")

# STEP 2: Then, get the top 10 predictions.
test_user1.createOrReplaceTempView("test_user1")
test_user1_top10 = spark.sql("""
    SELECT *
    FROM test_user1
    ORDER BY prediction DESC
    LIMIT 10
""")


test_user1_top10.show()

+------+--------+---------+-----------+
|userid|artistid|playcount| prediction|
+------+--------+---------+-----------+
|   321| 1275996|        1| 0.10846133|
|   321|    2178|        3| 0.08306771|
|   321|    1991|       12| 0.07769287|
|   321|    4951|      262|0.037496418|
|   321|     304|       19|0.024802346|
|   321| 1028974|        3|0.024409773|
|   321| 1028717|        1|0.017516244|
|   321|    1365|        8|0.017342392|
|   321| 1233802|        7|0.016580783|
|   321|     378|        7|0.016491838|
+------+--------+---------+-----------+



**TASK 4 (1 PT)**  
**Briefly explain Recall@K in the context of recommender systems**

Recall@k is like the concept of recall in class imbalance problems which is the number of True Positives (TP) that are correctly predicted, e.g., TP / (TP + FN).  What's different for recommendation problems is that the equivalent of True Positives (the numerator) is the top number (k) of the total predicted items of which the user did interact with and which the model correctly predicted as such, and the denominator is this number plus all the other items that the user interacted with.

In [8]:
# TASK 5 (1 PT)
# Calculate Recall@K for Test User 1

# This is asking, "Of all the artists that the user listened to, how many of the top-k predictions did the model correctly predict?"
# Mathematically, the formula is:
#
#              # artists the user listened to that are predicted in the top k
#  Recall@K = ----------------------------------------------------------------
#                     # all of the artists the user listened to


# One way to do this is with SQL.
# I'll stick with k=10 for this.

# This requires, querying the test data.
testDF.createOrReplaceTempView("testDF")

# STEP 1: DENOMINATOR.
# Get the total rows for userid = 321.
all_items = spark.sql("""
SELECT *
FROM testDF
WHERE userid = 321
""")

# STEP 2: NUMERATOR.
# Get the total of these that were in the predicted set; these are the true positives (tp).
# This part uses the top k=10 from TASK 3.
all_items.createOrReplaceTempView("all_items")
test_user1_top10.createOrReplaceTempView("test_user1_top10")
tp = spark.sql("""
SELECT *
FROM test_user1_top10 AS t
JOIN all_items AS a
ON t.artistid = a.artistid
""")

# STEP 3: CALCULATE the Recall@K.
tp.createOrReplaceTempView("tp")
recall_k = spark.sql("""
SELECT 
    ROUND(
            (SELECT COUNT(*) FROM tp) /
            (SELECT COUNT(*) FROM all_items)
            ,4
        ) AS RecallK
""")

# I had trouble extracting the value from the recall_k DataFrame so I had MS Copilot give me this syntax.
# I just don't do enough straight up Python coding for indexing to have sunk in.
print(f"The Recall@K for userid=321 is: {recall_k.collect()[0]['RecallK']}")

The Recall@K for userid=321 is: 0.2703


In [9]:
# ALS MODEL 2: For this model, you will use hyperparameter tuning on the validation set to select the best values for
# (rank, maxIter, regParam, alpha) to maximize Recall@K

# Import the necessary library.
from pyspark.ml.tuning import ParamGridBuilder

# Build the hyperparameter tuning grid.
# I had Gemini help recommend values for these parameters.
# I added comments to explain my understanding of why to use these values.
paramGrid = (
        ParamGridBuilder()
    
        # Specify different number of hidden features to look for.
        .addGrid(als_spec.rank, [10, 20, 40])
    
        # Specify how many times to loop through the data.
        .addGrid(als_spec.maxIter, [10])
    
        # Specify the regularization penalty; 0.01 is flexible and 0.1 is stricter (less-overfitting).
        .addGrid(als_spec.regParam, [0.01, 0.1])
    
        # Specify how much weight is placed on the playcounts data; 1.0 a little weight, 10.0 a lot of weight.
        .addGrid(als_spec.alpha, [1.0, 10.0])
    
        # Build the tuning grid.
        .build()
    )

print("The paramGrid has been built.")

The paramGrid has been built.


In [10]:
# TASK 6 (2 PTS)
# Report the results including best values for (rank, maxIter, regParam, alpha) along with the Recall@K from the validation set

# Import the necessary libraries.
from pyspark.ml.tuning import CrossValidator
from pyspark.ml.evaluation import RegressionEvaluator

# This evaluator will evaluate RMSE to tune parameters and then Recall@K will be calculated later on the validation set.
evaluator_spec = RegressionEvaluator(
    predictionCol="prediction",  # Use the ALS predictions.
    labelCol="playcount",        # Use the playcount data as the "truth" from the held-out data set for each CV fold.
    metricName="rmse",           # Specifies to optimize RMSE.
)

# Specify the cross validation approach.
crossval = CrossValidator(
    estimator=als_spec,
    estimatorParamMaps=paramGrid,
    evaluator= evaluator_spec,
    numFolds=2
)

print("The evaluator and crossval specs are built.")

The evaluator and crossval specs are built.


In [11]:
# THIS STEP TAKES SOME TIME SO IT IS PUT INTO ITS OWN CELL.

# Time this step.
import time
t0 = time.time()

# Run cross-validation on the validation data, leveraging parallel processing.
cvModel_val = crossval.setParallelism(4).fit(valDF)

# Show how much elapsed.
print("train time:", time.time() - t0)

                                                                                ]

[415.313s][warning][gc,alloc] dag-scheduler-event-loop: Retried waiting for GCLocker too often allocating 524290 words


26/10/11 19:35:23 ERROR TorrentBroadcast: Store broadcast broadcast_911 fail, remove all pieces of the broadcast
26/10/11 19:35:23 ERROR Instrumentation: org.apache.spark.SparkException: Job aborted due to stage failure: Task serialization failed: java.lang.OutOfMemoryError: Java heap space
java.lang.OutOfMemoryError: Java heap space
	at java.base/java.nio.HeapByteBuffer.<init>(HeapByteBuffer.java:71)
	at java.base/java.nio.ByteBuffer.allocate(ByteBuffer.java:391)
	at org.apache.spark.broadcast.TorrentBroadcast$.$anonfun$blockifyObject$1(TorrentBroadcast.scala:364)
	at org.apache.spark.broadcast.TorrentBroadcast$.$anonfun$blockifyObject$1$adapted(TorrentBroadcast.scala:364)
	at org.apache.spark.broadcast.TorrentBroadcast$$$Lambda/0x00007f4d18ebc000.apply(Unknown Source)
	at org.apache.spark.util.io.ChunkedByteBufferOutputStream.allocateNewChunkIfNeeded(ChunkedByteBufferOutputStream.scala:87)
	at org.apache.spark.util.io.ChunkedByteBufferOutputStream.write(ChunkedByteBufferOutputStream.

Py4JJavaError: An error occurred while calling o2884.fit.
: org.apache.spark.SparkException: Job aborted due to stage failure: Task serialization failed: java.lang.OutOfMemoryError: Java heap space
java.lang.OutOfMemoryError: Java heap space
	at java.base/java.nio.HeapByteBuffer.<init>(HeapByteBuffer.java:71)
	at java.base/java.nio.ByteBuffer.allocate(ByteBuffer.java:391)
	at org.apache.spark.broadcast.TorrentBroadcast$.$anonfun$blockifyObject$1(TorrentBroadcast.scala:364)
	at org.apache.spark.broadcast.TorrentBroadcast$.$anonfun$blockifyObject$1$adapted(TorrentBroadcast.scala:364)
	at org.apache.spark.broadcast.TorrentBroadcast$$$Lambda/0x00007f4d18ebc000.apply(Unknown Source)
	at org.apache.spark.util.io.ChunkedByteBufferOutputStream.allocateNewChunkIfNeeded(ChunkedByteBufferOutputStream.scala:87)
	at org.apache.spark.util.io.ChunkedByteBufferOutputStream.write(ChunkedByteBufferOutputStream.scala:75)
	at net.jpountz.lz4.LZ4BlockOutputStream.flushBufferedData(LZ4BlockOutputStream.java

[418.616s][warning][gc,alloc] broadcast-exchange-43: Retried waiting for GCLocker too often allocating 1048578 words


[Stage 9526:>                                                     (0 + 16) / 17]]

[460.309s][warning][gc,alloc] Executor task launch worker for task 16.0 in stage 9526.0 (TID 36489): Retried waiting for GCLocker too often allocating 3 words


26/10/11 19:36:08 ERROR Executor: Exception in task 16.0 in stage 9526.0 (TID 36489)
java.lang.OutOfMemoryError: Java heap space
	at scala.collection.mutable.ListBuffer.addOne(ListBuffer.scala:117)
	at scala.collection.mutable.ListBuffer.addOne(ListBuffer.scala:40)
	at scala.collection.mutable.Growable.$plus$eq(Growable.scala:36)
	at scala.collection.mutable.Growable.$plus$eq$(Growable.scala:36)
	at scala.collection.mutable.AbstractBuffer.$plus$eq(Buffer.scala:314)
	at org.apache.spark.sql.catalyst.expressions.GeneratedClass$SpecificSafeProjection.MapObjects_0$(Unknown Source)
	at org.apache.spark.sql.catalyst.expressions.GeneratedClass$SpecificSafeProjection.apply(Unknown Source)
	at org.apache.spark.sql.catalyst.encoders.ExpressionEncoder$Deserializer.apply(ExpressionEncoder.scala:89)
	at org.apache.spark.sql.catalyst.expressions.ScalaUDF.$anonfun$scalaConverter$2(ScalaUDF.scala:171)
	at org.apache.spark.sql.catalyst.expressions.ScalaUDF$$Lambda/0x00007f4d1946ab88.apply(Unknown Sourc

In [ ]:
# Extract the best performing model and get the rank from it.
best_model_val = cvModel_val.bestModel
best_rank = best_model_val.rank

# To get the hyperparamters (maxIter, regParam, alpha) I asked Gemini for help getting the best values.
# I didn't see anything in the prior material that explained how to get at these.

# Create a clean dictionary mapping parameter names to their best values
best_params_dict = {param.name: value for param, value in best_params.items()}

# Assign them to explicit variables for later use in retraining the train + validation data.
best_maxIter = best_params_dict.get("maxIter")
best_regParam = best_params_dict.get("regParam")
best_alpha = best_params_dict.get("alpha")

# Dispaly the best values.
print("--- Best Hyperparameters Found ---")
print(f"rank = {best_rank}")
print(f"maxIter = {best_maxIter}")
print(f"regParam = {best_regParam}")
print(f"alpha = {best_alpha}")

print("\n--- Best RMSE ---")
print(f"Best Validation RMSE: {min(cvModel_val.avgMetrics):.4f}")

In [ ]:
# Retrain the best configuration on train + validation data.

# Combine trainDF and valDF.
tvDF = trainDF.union(valDF)
tvDF = tvDF.cache()

# Update the model spec to use the best config.
als_spec_tuned = ALS(
    userCol="userid",
    itemCol="artistid",
    ratingCol="playcount",
    implicitPrefs=True,
    coldStartStrategy="drop",
    rank=best_model_val.rank,
    maxIter=best_maxIter,
    regParam=best_regParam,
    alpha=best_alpha
)

print("The tuned als_spec has been built.")

In [ ]:
# THIS STEP TAKES A FEW SECONDS...PUT INTO ITS OWN CELL IN CASE IT RUNS LONG.

# Time this step.
t0 = time.time()

# Fit the final model on the combined train and validation data.
als_model_tuned = als_spec_tuned.fit(tvDF) 

# Show how much elapsed.
print("train time:", time.time() - t0)

print("The final tuned model has been fit using the training and validation data.")

In [12]:
# TASK 7 (1 PT)
# Calculate Recall@K on the test set and report the value.

# For this, I will skip trying to integrate SQL and instead use PySpark.
# I had to get help from Gemini because I struggled a lot to piece this together.

# Import the necesssary functions library.
from pyspark.sql import functions as F

# Get the unique set of userids; cache it to help with performance.
test_users = testDF.select("userid").distinct().cache()

# Generate the top 10 recommendations from this subset.
test_top10_recs = als_model_tuned.recommendForUserSubset(test_users, 10)

# Explode recommendations into a flat DataFrame of predicted items; cache for performance.
test_top10DF = test_top10_recs.select(
    F.col("userid"),
    F.explode(F.col("recommendations")).alias("rec")
).select(
    F.col("userid"),
    F.col("rec.artistid").alias("artistid")
).cache()

# STEP 1: DENOMINATOR.
# Count the total number of artists for each user.
all_counts = testDF.groupBy("userid").agg(F.count("artistid").alias("total_actual"))

# STEP 2: NUMERATOR
# Count True Positives (tp) using an inner join.
# Get the total of these that were in the predicted set; these are the true positives (tp).
tp_all_count = test_top10DF.join(testDF, on=["userid", "artistid"], how="inner").groupBy("userid").agg(F.count("artistid").alias("tp_count"))

# STEP 3: CALCULATE the Recall@K.
recall_df = all_counts.join(tp_all_count, on="userid", how="left").na.fill(0, subset=["tp_count"]).withColumn(
    "recall_k", F.col("tp_count") 
    / F.col("total_actual")
)
# Calculate the average over all users.
mean_recall_k = recall_df.agg(F.avg("recall_k")).collect()[0][0]

print(f"Mean Recall@K across the test set is: {mean_recall_k:.4f}")

NameError: name 'als_model_tuned' is not defined

In [ ]:
# TASK 8 (1 PT)
# Calculate Recall@K for Test User 1 
# Justify the result, showing the recommendations and user activity. Did hyperparameter tuning seem to help for this user?

# For this, I will skip trying to integrate SQL and instead use PySpark.

# Import the necesssary functions library.
from pyspark.sql import functions as F

# Get all the recommendations for userid = 321.
user321_recs = spark.createDataFrame([(321,)], ["userid"])

# Generate the top 10 recommendations from this subset.
user1_recs_top10 = final_model.recommendForUserSubset(user321_recs, 10)

# Explode recommendations into a flat DataFrame of predicted items.
top10DF = user1_recs_top10.select(
    F.col("userid"),
    F.explode(F.col("recommendations")).alias("rec")
).select(
    F.col("userid"),
    F.col("rec.artistid").alias("artistid")
)

# STEP 1: DENOMINATOR.
# Get all rows for userid = 321 from test data.
testDF_user321 = testDF.filter(F.col("userid") == 321)
testDF_user321_count = testDF_user321.count()

# STEP 2: NUMERATOR
# Count True Positives (tp) using an inner join.
# Get the total of these that were in the predicted set; these are the true positives (tp).
tp_count = top10DF.join(testDF_user321, on=["userid", "artistid"], how="inner").count()

# STEP 3: CALCULATE the Recall@K.
recall_k = tp_count / testDF_user321_count if testDF_user321_count > 0 else 0.0

print(f"The Recall@K for userid=321 is: {recall_k:.4f}")

In [ ]:
# The results above suggest the model did not recommend any of the artists that userid=321 actually listened to.
# Reviewing the top 10 recommendations and the artists in the test data set confirms this.
# Per output below (visual inspect) there is no overlap between between the predictions and the test data for userid=321.

print("--- Top 10 Recommendations ---")
top10DF.show()

print("--- Actual Test Set Items for User 321 ---")
testDF_user321.show()

**Conclusion.**
The hyperparameter tuning did NOT improve recommendations for this user.